In [1]:
!nvidia-smi

Mon Aug 17 12:54:27 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### **1. Install Dependencies**

In [14]:
!pip install -q vllm
!pip install -q pyngrok
!pip install -q torch transformers
!pip install -U huggingface_hub

In [16]:
from pyngrok import ngrok

# Replace with your actual token
ngrok_token = "3I2n7d3rirH7VMZ2EskIJLmgs8T_26MVNuVNVCob3x34Rp4Js"  # ← PASTE YOUR TOKEN HERE

ngrok.set_auth_token(ngrok_token)
print("✅ Ngrok authenticated!")

✅ Ngrok authenticated!


In [35]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.13.0+cu130
CUDA: 13.0
GPU: Tesla T4


In [19]:
!pip uninstall -y torchaudio

Found existing installation: torchaudio 2.11.0+cu128
Uninstalling torchaudio-2.11.0+cu128:
  Successfully uninstalled torchaudio-2.11.0+cu128


In [21]:
import vllm

print("vLLM:", vllm.__version__)

vLLM: 0.27.1


### **2. Download the model**

In [ ]:
# LLAMA 2 :

# !hf download meta-llama/Llama-2-7b-hf \
#     --local-dir /content/models/Llama-2-7b-hf

In [37]:
# QWEN-2.5B-3B :

!hf download Qwen/Qwen2.5-3B-Instruct \
    --local-dir /content/models/Qwen2.5-3B-Instruct

Hint: The `hf-cli` skill is not installed. Run `hf skills add -g --claude` to teach your AI agents how to use the `hf` CLI.
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 12 files:   0% 0/12 [00:00<?, ?it/s]Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.

Reconstructing (incomplete total...):   0% 0.00/1.52k [00:00<?, ?B/s]         

Fetching 12 files:   8% 1/12 [00:00<00:01,  5.80it/s]
Reconstructing (incomplete total...):  24% 1.52k/6.45k [00:00<00:00, 9.49kB/s]
Reconstructing (incomplete total...):  47% 6.45k/13.8k [00:00<00:00, 9.49kB/s]

Fetching 12 files:  50% 6/12 [00:00<00:00, 24.46it/s]

Reconstructing (incomplete total...): 100% 13.8k/13.8k [00:20<00:00, 9.49kB/s]

Fetching 12 files: 100% 12/12 [00:33<00:00,  2.77s/it]
Download complete: 100% 13.8k/13.8k [00:33<00:00, 9.47kB/s]
Reconstruction complete: 100% 13.8k/13.8k [00:33<00:00, 9.49kB/s]    

In [38]:
!ls -lh /content/models/Qwen2.5-3B-Instruct

total 5.8G
-rw-r--r-- 1 root root  661 Aug 17 13:32 config.json
-rw-r--r-- 1 root root  242 Aug 17 13:32 generation_config.json
-rw-r--r-- 1 root root 7.3K Aug 17 13:32 LICENSE
-rw-r--r-- 1 root root 1.6M Aug 17 13:32 merges.txt
-rw-r--r-- 1 root root 3.7G Aug 17 13:32 model-00001-of-00002.safetensors
-rw-r--r-- 1 root root 2.1G Aug 17 13:32 model-00002-of-00002.safetensors
-rw-r--r-- 1 root root  35K Aug 17 13:32 model.safetensors.index.json
-rw-r--r-- 1 root root 4.9K Aug 17 13:32 README.md
-rw-r--r-- 1 root root 7.2K Aug 17 13:32 tokenizer_config.json
-rw-r--r-- 1 root root 6.8M Aug 17 13:32 tokenizer.json
-rw-r--r-- 1 root root 2.7M Aug 17 13:32 vocab.json


### **4. Start vLLM using the local model**

In [39]:
!python -m vllm.entrypoints.openai.api_server \
    --model /content/models/Qwen2.5-3B-Instruct \
    --host 0.0.0.0 \
    --port 8000 \
    --gpu-memory-utilization 0.85

(APIServer pid=13806) INFO 08-17 13:33:21 [api_utils.py:345] 
(APIServer pid=13806) INFO 08-17 13:33:21 [api_utils.py:345]        █     █     █▄   ▄█
(APIServer pid=13806) INFO 08-17 13:33:21 [api_utils.py:345]  ▄▄ ▄█ █     █     █ ▀▄▀ █  version 0.27.1
(APIServer pid=13806) INFO 08-17 13:33:21 [api_utils.py:345]   █▄█▀ █     █     █     █  model   /content/models/Qwen2.5-3B-Instruct
(APIServer pid=13806) INFO 08-17 13:33:21 [api_utils.py:345]    ▀▀  ▀▀▀▀▀ ▀▀▀▀▀ ▀     ▀
(APIServer pid=13806) INFO 08-17 13:33:21 [api_utils.py:345] 
(APIServer pid=13806) INFO 08-17 13:33:21 [api_utils.py:273] non-default args: {'host': '0.0.0.0', 'model': '/content/models/Qwen2.5-3B-Instruct', 'gpu_memory_utilization': 0.85}
(APIServer pid=13806) Traceback (most recent call last):
(APIServer pid=13806)   File "<frozen runpy>", line 198, in _run_module_as_main
(APIServer pid=13806)   File "<frozen runpy>", line 88, in _run_code
(APIServer pid=13806)   File "/usr/local/lib/python3.12/dist-packages/vllm/ent

In [40]:
!curl http://localhost:8000/v1/models

{"object":"list","data":[{"id":"Qwen/Qwen2.5-3B-Instruct","object":"model","created":1786973612,"owned_by":"vllm","root":"Qwen/Qwen2.5-3B-Instruct","parent":null,"max_model_len":32768,"permission":[{"id":"modelperm-be799a5cf2c335e6","object":"model_permission","created":1786973612,"allow_create_engine":false,"allow_sampling":true,"allow_logprobs":true,"allow_search_indices":false,"allow_view":true,"allow_fine_tuning":false,"organization":"*","group":null,"is_blocking":false}]}]}